# 05 — Stationarity Analysis & Automated Differencing

**Mục tiêu của phần này:**
1. **Kiểm định tính dừng đa biến:** Mở rộng việc kiểm tra tính dừng (Stationarity) bằng Augmented Dickey-Fuller (ADF) cho toàn bộ các đặc trưng mới trích xuất (RMS, Variance, PTP, Line Length).
2. **Tự động lấy sai phân (Automated Differencing):** Viết hàm tự động lấy sai phân bậc 1 ($d=1$) hoặc bậc 2 ($d=2$) cho đến khi chuỗi tín hiệu đạt được tính dừng (p-value < 0.05). Việc này đảm bảo tất cả các feature đều thỏa mãn điều kiện đầu vào của mô hình ARIMA/Hồi quy chuỗi thời gian.

**Đầu vào:** File `ds003029_forecasting_features.csv`
**Đầu ra:** File `ds003029_stationary_features.csv`

In [1]:
import pandas as pd
from statsmodels.tsa.stattools import adfuller
from pathlib import Path
import sys

# Load đường dẫn (nếu bạn dùng thư viện nội bộ)
ws = Path.cwd().resolve()
src_dir = (ws / 'src' if (ws / 'src').exists() else ws.parent / 'src').resolve()
sys.path.insert(0, str(src_dir))
from ds003029_eda.paths import get_paths
paths = get_paths()

# Load dữ liệu từ Notebook 04
input_file = paths.outputs_dir / "ds003029_forecasting_features_multirun.csv"
df = pd.read_csv(input_file)

def check_stationarity_and_difference(df, col, max_d=2):
    """Kiểm tra ADF và lấy sai phân an toàn theo từng run_id"""
    s = df[col].dropna()
    # Check d=0 trên toàn bộ chuỗi
    if adfuller(s)[1] < 0.05:
        return df[col], 0
        
    for d in range(1, max_d + 1):
        # Tính sai phân nội bộ từng file để không bị vượt ranh giới
        diff_series = df.groupby('run_id')[col].diff(d)
        s_diff = diff_series.dropna()
        if adfuller(s_diff)[1] < 0.05:
            return diff_series, d
            
    return df.groupby('run_id')[col].diff(max_d), max_d

# 1. Xử lý tính dừng cho các cột đặc trưng
df_stationary = df[['run_id', 't_mid_s']].copy()
cols_to_transform = [c for c in df.columns if c not in ['run_id', 't_mid_s']]

d_values = {}
for col in cols_to_transform:
    stat_series, d_val = check_stationarity_and_difference(df, col)
    df_stationary[col] = stat_series
    d_values[col] = d_val

print("Bậc sai phân (d) cần thiết để chuỗi dừng:")
for col, d in d_values.items():
    print(f" - {col}: d = {d}")

# Loại bỏ các hàng bị NaN ở đầu mỗi file do quá trình sai phân tạo ra
df_stationary = df_stationary.dropna().reset_index(drop=True)

# 2. Lưu file Output
out_file = paths.outputs_dir / "ds003029_stationary_features_multirun.csv"
df_stationary.to_csv(out_file, index=False)
print(f"✅ Đã lưu bộ dữ liệu stationary tại {out_file.name} với {len(df_stationary)} bản ghi.")

Bậc sai phân (d) cần thiết để chuỗi dừng:
 - rms: d = 1
 - variance: d = 1
 - ptp: d = 1
 - line_length: d = 1
 - rms_lag1: d = 1
 - rms_lag2: d = 1
 - variance_lag1: d = 1
 - variance_lag2: d = 1
 - ptp_lag1: d = 1
 - ptp_lag2: d = 1
 - line_length_lag1: d = 1
 - line_length_lag2: d = 1
✅ Đã lưu bộ dữ liệu stationary tại ds003029_stationary_features_multirun.csv với 5222 bản ghi.
